In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw

# Dataset root directory
DATASET_DIR = Path("data")

# Dataset splits
TRAIN_DIR = DATASET_DIR / "train"
VALID_DIR = DATASET_DIR / "valid"
TEST_DIR = DATASET_DIR / "test"

# COCO annotation files
TRAIN_ANNOTATIONS = TRAIN_DIR / "_annotations.coco.json"
VALID_ANNOTATIONS = VALID_DIR / "_annotations.coco.json"
TEST_ANNOTATIONS = TEST_DIR / "_annotations.coco.json"

# Check dataset paths
print("Working directory:", Path.cwd())
print("Training dataset found:", TRAIN_ANNOTATIONS.exists())
print("Validation dataset found:", VALID_ANNOTATIONS.exists())
print("Test dataset found:", TEST_ANNOTATIONS.exists())

In [ ]:
# Load training annotations
with open(TRAIN_ANNOTATIONS, "r") as f:
    train_data = json.load(f)

# Extract images and annotations
images = train_data["images"]
annotations = train_data["annotations"]

print(f"Number of images: {len(images)}")
print(f"Number of annotations: {len(annotations)}")

In [ ]:
def create_binary_mask(image_info, annotations):
    """
    Create a binary segmentation mask from COCO polygon annotations.

    Args:
        image_info: Dictionary containing image metadata.
        annotations: List of annotations associated with the image.

    Returns:
        Binary mask as a NumPy array (0 = background, 1 = annotated region).
    """

    width = image_info["width"]
    height = image_info["height"]

    # Create an empty black mask
    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)

    # Draw all annotated polygons
    for annotation in annotations:
        for polygon in annotation["segmentation"]:

            points = [
                (polygon[i], polygon[i + 1])
                for i in range(0, len(polygon), 2)
            ]

            draw.polygon(points, fill=1)

    return np.array(mask, dtype=np.bool)

In [ ]:
import random

# Select 10 random images
random.seed(42)
selected_images = random.sample(images, 10)

# Group annotations by image ID
annotations_by_image = {}

for annotation in annotations:
    image_id = annotation["image_id"]

    if image_id not in annotations_by_image:
        annotations_by_image[image_id] = []

    annotations_by_image[image_id].append(annotation)

# Create a figure with 10 rows and 2 columns
fig, axes = plt.subplots(10, 2, figsize=(12, 40))

for i, image_info in enumerate(selected_images):

    # Load original image
    image_path = TRAIN_DIR / image_info["file_name"]
    image = Image.open(image_path).convert("RGB")

    # Get corresponding annotations
    image_annotations = annotations_by_image.get(
        image_info["id"], []
    )

    # Generate binary mask
    mask = create_binary_mask(image_info, image_annotations)

    # Verify that image and mask have the same dimensions
    assert mask.shape == (image.height, image.width)

    # Display original image
    axes[i, 0].imshow(image)
    axes[i, 0].set_title(f"Original Image {i + 1}")
    axes[i, 0].axis("off")

    # Display binary mask
    axes[i, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title(f"Binary Mask {i + 1}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score
)

def load_split(split_dir):
    """Load images and group annotations by image ID."""

    with open(split_dir / "_annotations.coco.json", "r") as f:
        data = json.load(f)

    annotations_by_image = defaultdict(list)

    for annotation in data["annotations"]:
        annotations_by_image[annotation["image_id"]].append(annotation)

    return data["images"], annotations_by_image


train_images, train_annotations_by_image = load_split(TRAIN_DIR)
valid_images, valid_annotations_by_image = load_split(VALID_DIR)
test_images, test_annotations_by_image = load_split(TEST_DIR)

print("Training:", len(train_images))
print("Validation:", len(valid_images))
print("Test:", len(test_images))

In [ ]:
def sample_pixels(image_infos, annotations_by_image, images_dir,n_pixels, rng):

    features = []
    labels = []

    for image_info in image_infos:

        image_path = images_dir / image_info["file_name"]

        with Image.open(image_path) as img:
            image = np.asarray(img.convert("L"))

        mask = create_binary_mask(
            image_info,
            annotations_by_image.get(image_info["id"], [])
        )

        assert image.shape == mask.shape

        height, width = image.shape

        ys = rng.integers(0, height, size=n_pixels)
        xs = rng.integers(0, width, size=n_pixels)

        features.append(
            image[ys, xs].astype(np.float32).reshape(-1, 1) / 255.0
        )

        labels.append(mask[ys, xs].astype(np.uint8))

    return np.vstack(features), np.concatenate(labels)


N_PIXELS = 100
rng = np.random.default_rng(42)

features_tr, labels_tr = sample_pixels(
    train_images, train_annotations_by_image,TRAIN_DIR, N_PIXELS, rng
)

features_val, labels_val = sample_pixels(
    valid_images, valid_annotations_by_image,VALID_DIR, N_PIXELS, rng
)

features_te, labels_te = sample_pixels(
    test_images, test_annotations_by_image,TEST_DIR, N_PIXELS, rng
)

print("Train:", features_tr.shape, labels_tr.shape)
print("Validation:", features_val.shape, labels_val.shape)
print("Test:", features_te.shape, labels_te.shape)

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)

clf.fit(features_tr, labels_tr)

print("Model trained successfully!")

In [ ]:
def evaluate_classifier(model, X, y, split_name):

    predicted = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    accuracy = accuracy_score(y, predicted)
    auc = roc_auc_score(y, probabilities)

    print(f"{split_name} Accuracy: {accuracy:.4f}")
    print(f"{split_name} AUC: {auc:.4f}")

    cm = confusion_matrix(y, predicted, labels=[0, 1])
    print(cm)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Background", "Annotated area"]
    )

    disp.plot(cmap="Blues")
    plt.title(f"{split_name} Confusion Matrix")
    plt.show()


evaluate_classifier(clf, features_val, labels_val, "Validation")
evaluate_classifier(clf, features_te, labels_te, "Test")

In [ ]:
# Logistic Regression with balanced class weights
clf_balanced = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

clf_balanced.fit(features_tr, labels_tr)

# Evaluate on validation and test sets
evaluate_classifier(
    clf_balanced, features_val, labels_val, "Validation Balanced"
)

evaluate_classifier(
    clf_balanced, features_te, labels_te, "Test Balanced"
)

In [ ]:
# Visualize predicted masks on test images

selected_test_images = test_images[:3]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i, image_info in enumerate(selected_test_images):

    # Load original grayscale image
    image_path = TEST_DIR / image_info["file_name"]

    with Image.open(image_path) as img:
        image = np.asarray(img.convert("L"))

    # Load ground truth mask
    mask = create_binary_mask(
        image_info,
        test_annotations_by_image.get(image_info["id"], [])
    )

    height, width = image.shape

    # Prepare all pixels for classification
    X = image.astype(np.float32).reshape(-1, 1) / 255.0

    # Predict every pixel
    prediction_standard = clf.predict(X).reshape(height, width)
    prediction_balanced = clf_balanced.predict(X).reshape(height, width)

    # Display results
    axes[i, 0].imshow(image, cmap="gray")
    axes[i, 0].set_title("Original Image")

    axes[i, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground Truth")

    axes[i, 2].imshow(prediction_standard, cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title("Standard Prediction")

    axes[i, 3].imshow(prediction_balanced, cmap="gray", vmin=0, vmax=1)
    axes[i, 3].set_title("Balanced Prediction")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

# print balanced results percs

### Results discussion — Pixel-based segmentation

The standard Logistic Regression predicts every pixel as background due to the strong class imbalance. As a result, it achieves high accuracy but fails to identify any annotated region.

The balanced Logistic Regression detects more positive pixels, but it also produces a large number of false positives, resulting in low precision.

These results show the limitations of using only a single pixel's intensity for image segmentation. Pixels belonging to different regions may have similar intensity values, making them difficult to distinguish without additional spatial information.

In the next experiment, we will use image patches to include information from neighboring pixels and evaluate whether this improves segmentation performance.

In [ ]:
def pick_random_centers(mask, size=100, ignore=0):
    mask_ignored = mask.copy()
    mask_ignored[:ignore,:]=False
    mask_ignored[-ignore:,:]=False
    mask_ignored[:,:ignore]=False
    mask_ignored[:,-ignore:]=False
    rs, cs = np.where(mask_ignored)
    ix = np.random.randint(len(rs), size=size)
    return rs[ix], cs[ix]

for tim in train_images[:10]:
  image_path = TRAIN_DIR / tim["file_name"]
  with Image.open(image_path) as img:
      image = np.asarray(img.convert("L"))
  # Load ground truth mask
  mask = create_binary_mask(
      tim,
      test_annotations_by_image.get(tim["id"], [])
  )
  plt.imshow(image, cmap="gray")

  patch_size = 32//2

  c0_patches = []
  c0_labels = []
  rs,cs = pick_random_centers(mask, size=10, ignore=patch_size*2)
  plt.plot(cs, rs, 'b.')
  for r,c in zip(rs,cs):
      c0_patches.append(image[r-patch_size:r+patch_size, c-patch_size:c+patch_size])
      c0_labels.append(mask[r,c])

  c1_patches = []
  c1_labels = []
  rs,cs = pick_random_centers(~mask, size=10, ignore=patch_size*2)
  plt.plot(cs, rs, 'w.')
  for r,c in zip(rs,cs):
      c1_patches.append(image[r-patch_size:r+patch_size, c-patch_size:c+patch_size])
      c1_labels.append(mask[r,c])

  fig, axes = plt.subplots(2, 10,figsize=(15, 3.5))
  for (i, p) in enumerate(c0_patches):
    axes[0, i].imshow(p, cmap="gray", vmin=0, vmax=255)
  for (i, p) in enumerate(c1_patches):
    axes[1, i].imshow(p, cmap="gray", vmin=0, vmax=255)

  for ax in axes.flat:
      ax.set_xticks([])
      ax.set_yticks([])

  axes[0, 0].set_ylabel("C0", fontsize=12,  labelpad=10)
  axes[1, 0].set_ylabel("C1", fontsize=12,  labelpad=10)

  # plt.tight_layout()
  plt.show()
